# Inferenza per la competizione

In [1]:
import os
import json
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import create_tracker
from tqdm import tqdm

# Modificare i percorsi secondo necessità
DATASET_ROOT = "./SIMULATOR/lecture_example_from_training/test_set/videos/"  
OUTPUT_DIR    = "./SIMULATOR/lecture_example_from_training/Predictions_folder/"
CONFIG_PATH   = "./tracker_params.yaml"

MODEL_WEIGHTS = "./detection_models/ft_1920_yolo11m.pt"
REID_WEIGHTS  = "./reid_models/ft_osnet_ain_x1_0_imagenet.pt"

ID_GROUP = 10

DEVICE = "cuda:0"
CONF_THRES = 0.15
IOU_THRES  = 0.60
IMG_W = 1920
IMG_H = 1080

os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_roi_rect(roi_data, w, h):
    return (
        int(roi_data["x"] * w),
        int(roi_data["y"] * h),
        int(roi_data["width"] * w),
        int(roi_data["height"] * h),
    )

def in_roi(fx, fy, roi_rect):
    rx, ry, rw, rh = roi_rect
    return (
        rx <= fx <= rx + rw and
        ry <= fy <= ry + rh
    )

print(f"Caricamento Modello YOLO")
model = YOLO(MODEL_WEIGHTS, task="detect")

sequences = sorted([d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))])
print(f"Trovate {len(sequences)} sequenze da elaborare.")

for seq_name in sequences:
    print(f"\nElaborazione sequenza: {seq_name}")

    # Caricamento ROI
    roi_path = os.path.join(DATASET_ROOT, seq_name, "roi.json")
    if not os.path.exists(roi_path):
        print(f"File ROI non trovato: {roi_path}.")
        continue
    with open(roi_path, "r") as f:
        roi_data = json.load(f)
    roi1 = get_roi_rect(roi_data["roi1"], IMG_W, IMG_H)
    roi2 = get_roi_rect(roi_data["roi2"], IMG_W, IMG_H)

    base_img_path = os.path.join(DATASET_ROOT, seq_name, "img1")
    
    # Output files 
    track_path = os.path.join(OUTPUT_DIR, f"tracking_{seq_name}_{ID_GROUP}.txt")
    beh_path   = os.path.join(OUTPUT_DIR, f"behavior_{seq_name}_{ID_GROUP}.txt")

    if not os.path.exists(base_img_path):
        print(f"Cartella immagini non trovata: {base_img_path}.")
        continue
    
    # Inizializzazione Tracker
    tracker = create_tracker(
        tracker_type="botsort",
        tracker_config=Path(CONFIG_PATH),
        reid_weights=REID_WEIGHTS,
        device=DEVICE,
        half=True,
    )

    frames = sorted([
        os.path.join(base_img_path, f)
        for f in os.listdir(base_img_path)
        if f.endswith((".jpg", ".png"))
    ])
    
    if not frames:
        print("Nessun frame trovato.")
        continue

    # Elaborazione frame
    with open(track_path, "w") as f_track, open(beh_path, "w") as f_beh:
        
        for frame_id, frame_path in enumerate(tqdm(frames, desc=f"Video {seq_name}"), start=1):
            
            frame = cv2.imread(frame_path)
            if frame is None: continue

            count_roi1 = 0
            count_roi2 = 0

            # Detetction
            result = model(frame, imgsz=1920, conf=CONF_THRES, iou=IOU_THRES, classes=[0], device=DEVICE, verbose=False)[0]

            detections = []
            if result.boxes is not None:
                boxes = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()
                clss  = result.boxes.cls.cpu().numpy()
                for (x1, y1, x2, y2), conf, cls in zip(boxes, confs, clss):
                    detections.append([x1, y1, x2, y2, conf, cls])

            detections = np.array(detections) if len(detections) else np.empty((0, 6))

            # Tracking
            tracks = tracker.update(detections, frame)

            # Scrittura risultati
            for track in tracks:
                x1, y1, x2, y2 = map(int, track[0:4])
                tid = int(track[4])
                w, h = x2 - x1, y2 - y1
                
                f_track.write(f"{frame_id},{tid},{x1},{y1},{w},{h}\n")

                # Check ROI
                fx = x1 + w // 2
                fy = y1 + h
                if in_roi(fx, fy, roi1): count_roi1 += 1
                if in_roi(fx, fy, roi2): count_roi2 += 1

            f_beh.write(f"{frame_id},1,{count_roi1}\n")
            f_beh.write(f"{frame_id},2,{count_roi2}\n")

    print(f"Completato: {seq_name}")


SUCCESS  | BotSort: det_thresh=0.15, max_age=60, max_obs=61, min_hits=3, iou_threshold=0.6, per_class=None, asso_func=iou, track_high_thresh=0.25, track_low_thresh=0.1, new_track_thresh=0.45, track_buffer=60, match_thresh=0.7, proximity_thresh=0.9, appearance_thresh=0.2, cmc_method=ecc, frame_rate=25, fuse_first_associate=True, with_reid=True, fuse_score=True, gmc_method=sparseOptFlow
INFO     | BoxMOT v15.1.0 🚀 Python-3.12.9 torch-2.5.1+cu121
CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
INFO     | reid_models\ft_osnet_ain_x1_0_imagenet.pt
INFO     | [PID 17020] Found existing ReID weights at reid_models\ft_osnet_ain_x1_0_imagenet.pt; skipping download.


Caricamento Modello YOLO
Trovate 5 sequenze da elaborare.

Elaborazione sequenza: 1


SUCCESS  | Loaded pretrained weights from reid_models\ft_osnet_ain_x1_0_imagenet.pt
Video 1: 100%|██████████| 750/750 [01:44<00:00,  7.20it/s]
SUCCESS  | BotSort: det_thresh=0.15, max_age=60, max_obs=61, min_hits=3, iou_threshold=0.6, per_class=None, asso_func=iou, track_high_thresh=0.25, track_low_thresh=0.1, new_track_thresh=0.45, track_buffer=60, match_thresh=0.7, proximity_thresh=0.9, appearance_thresh=0.2, cmc_method=ecc, frame_rate=25, fuse_first_associate=True, with_reid=True, fuse_score=True, gmc_method=sparseOptFlow
INFO     | BoxMOT v15.1.0 🚀 Python-3.12.9 torch-2.5.1+cu121
CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
INFO     | reid_models\ft_osnet_ain_x1_0_imagenet.pt
INFO     | [PID 17020] Found existing ReID weights at reid_models\ft_osnet_ain_x1_0_imagenet.pt; skipping download.
SUCCESS  | Loaded pretrained weights from reid_models\ft_osnet_ain_x1_0_imagenet.pt


Completato: 1

Elaborazione sequenza: 2


Video 2: 100%|██████████| 750/750 [01:40<00:00,  7.47it/s]
SUCCESS  | BotSort: det_thresh=0.15, max_age=60, max_obs=61, min_hits=3, iou_threshold=0.6, per_class=None, asso_func=iou, track_high_thresh=0.25, track_low_thresh=0.1, new_track_thresh=0.45, track_buffer=60, match_thresh=0.7, proximity_thresh=0.9, appearance_thresh=0.2, cmc_method=ecc, frame_rate=25, fuse_first_associate=True, with_reid=True, fuse_score=True, gmc_method=sparseOptFlow
INFO     | BoxMOT v15.1.0 🚀 Python-3.12.9 torch-2.5.1+cu121
CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
INFO     | reid_models\ft_osnet_ain_x1_0_imagenet.pt
INFO     | [PID 17020] Found existing ReID weights at reid_models\ft_osnet_ain_x1_0_imagenet.pt; skipping download.
SUCCESS  | Loaded pretrained weights from reid_models\ft_osnet_ain_x1_0_imagenet.pt


...

Completato: 3

Elaborazione sequenza: 4


Video 4: 100%|██████████| 750/750 [01:43<00:00,  7.22it/s]
SUCCESS  | BotSort: det_thresh=0.15, max_age=60, max_obs=61, min_hits=3, iou_threshold=0.6, per_class=None, asso_func=iou, track_high_thresh=0.25, track_low_thresh=0.1, new_track_thresh=0.45, track_buffer=60, match_thresh=0.7, proximity_thresh=0.9, appearance_thresh=0.2, cmc_method=ecc, frame_rate=25, fuse_first_associate=True, with_reid=True, fuse_score=True, gmc_method=sparseOptFlow
INFO     | BoxMOT v15.1.0 🚀 Python-3.12.9 torch-2.5.1+cu121
CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
INFO     | reid_models\ft_osnet_ain_x1_0_imagenet.pt
INFO     | [PID 17020] Found existing ReID weights at reid_models\ft_osnet_ain_x1_0_imagenet.pt; skipping download.
SUCCESS  | Loaded pretrained weights from reid_models\ft_osnet_ain_x1_0_imagenet.pt


Completato: 4

Elaborazione sequenza: 5


Video 5: 100%|██████████| 750/750 [01:40<00:00,  7.49it/s]

Completato: 5
